In [ ]:
# Install dependencies
!pip install playwright > /dev/null
!playwright install chromium > /dev/null
print("✓ Playwright installed!")

✓ Playwright installed!


In [ ]:
import asyncio
from playwright.async_api import async_playwright
from datetime import datetime
import sys
import traceback
import re

print("✓ Libraries imported successfully")

✓ Libraries imported successfully


In [ ]:
class StrandsAutomation:
    """
    Automated solver for NYT Strands puzzle, optimized based on OCR coordinate validation.
    Integrated with reliable navigation and screenshot capabilities for consistent zoom and visibility.
    """

    def __init__(self, headless=True, board_rows=8, board_cols=6, reset_position=(1059, 161)):
        self.headless = headless
        self.board_rows = board_rows
        self.board_cols = board_cols
        self.playwright = None
        self.browser = None
        self.cell_map = {}
        self.reset_position = reset_position  # Top-left cell coordinate for resetting pointer

        # Timing configuration (milliseconds)
        self.timing = {
            'initial_load': 3000,
            'tutorial_wait': 5000,
            'after_click': 800,
            'between_words': 4000,  # Increased delay to 4.0s to wait for board state update
            'mouse_move_step_delay': 150, # Delay between cell drags for stability
            'after_selection': 4000, # Wait 4.0s after word submission to ensure registration
            'mouse_reset_delay': 500 # Delay for mouse reset movement
        }

    async def start_playwright(self):
        """Initializes Playwright and launches the browser."""
        self.playwright = await async_playwright().start()
        self.browser = await self.playwright.chromium.launch(headless=self.headless)

    async def navigate_to_game(self, page, url="https://www.nytimes.com/games/strands"):
        """
        Navigates to the game in a fresh session and precisely completes the
        'How to Play' tutorial to ensure consistent visibility.
        """
        print(f"Navigating to {url}...")
        await page.goto(url, wait_until='domcontentloaded', timeout=60000)

        # 1. Locate and click the initial "Play" button using its unique test ID
        try:
            initial_play_button = page.locator('[data-testid="moment-btn-play"]')
            await initial_play_button.wait_for(state="visible", timeout=10000)
            print("Found and clicked the initial 'Play' button.")
            await initial_play_button.click()
        except Exception as e:
            print(f"ℹ No initial 'Play' button found or error: {e}")

        # 2. PRECISELY follow the 'Next -> Next -> Play' tutorial flow
        print("Waiting for the 'How to Play' tutorial...")
        try:
            next_button = page.locator('[data-testid="modal-help__next"]')
            await next_button.wait_for(state="visible", timeout=10000)
            print("Tutorial detected. Navigating...")

            # Click "Next" (first time)
            await next_button.click()
            print("Clicked 'Next' (1/2).")
            await page.wait_for_timeout(1000)

            # Click "Next" (second time)
            await next_button.click()
            print("Clicked 'Next' (2/2).")
            await page.wait_for_timeout(1000)

            # Click the final "Play" button inside the tutorial
            final_play_button = page.locator('[data-testid="modal-help__play"]')
            await final_play_button.click()
            print("Clicked final 'Play' button to start the game.")

        except Exception as e:
            print(f"ℹ Tutorial not found or already completed. Error: {e}")

        # Handle "You solved it yesterday" modal if it appears
        try:
            close_button = page.locator('[data-testid="results-modal-close"]')
            if await close_button.is_visible(timeout=2000):
                await close_button.click()
                print("✓ Closed previous results modal.")
                await page.wait_for_timeout(self.timing['after_click'])
        except Exception:
            pass

        # 3. Locate the main game container
        print("Waiting for the main game area to appear...")
        game_container_locator = page.locator('.pz-game-screen')
        await game_container_locator.wait_for(state="visible", timeout=10000)
        await page.wait_for_timeout(1000)
        print("✓ Game board is visible and ready.")

    async def take_screenshot(self, page, output_path="strands_screenshot.png"):
        """Takes a focused screenshot of the game area for consistent visibility."""
        game_container_locator = page.locator('.pz-game-screen')
        print(f"Taking a focused screenshot of the game area -> {output_path}")
        await game_container_locator.screenshot(path=output_path)
        print("Screenshot successful.")
        return output_path

    async def get_current_found_words(self, page):
        """Get the current number of theme words found."""
        try:
            progress_locator = page.locator('p:has-text("theme words found")')
            text = await progress_locator.inner_text(timeout=2000)
            match = re.search(r'(\d+) of \d+', text)
            if match:
                return int(match.group(1))
            else:
                return 0
        except Exception:
            return 0

    async def reset_mouse_position(self, page):
        """Reset mouse to top-left position to ensure clean state for next word."""
        sys.stdout.write(f"   Resetting mouse to top-left position {self.reset_position}\n")
        sys.stdout.flush()
        await page.mouse.move(self.reset_position[0], self.reset_position[1], steps=3)
        await page.wait_for_timeout(self.timing['mouse_reset_delay'])

    async def select_word(self, page, path):
        """
        Simulates a mouse drag across the path of coordinates (x, y pixel values).
        This method is the core automation step for selecting a word.
        """
        if len(path) < 2:
            sys.stdout.write(f"  ⚠ Path too short ({len(path)} cells), skipping\n")
            sys.stdout.flush()
            return False

        word_str = f"{len(path)}-letter word"
        sys.stdout.write(f"   Selecting: {word_str}\n")

        # Log the actual pixel path for debugging
        actual_pixel_path = ' → '.join(f"({x},{y})" for x, y in path)
        sys.stdout.write(f"     Path: {actual_pixel_path}\n")
        sys.stdout.flush()

        # 1. Move mouse to the starting cell
        start_cell = path[0]
        await page.mouse.move(start_cell[0], start_cell[1], steps=3)
        await page.wait_for_timeout(self.timing['mouse_move_step_delay'])

        # 2. Start drag
        await page.mouse.down()
        sys.stdout.write(f"     Start Drag: ({start_cell[0]},{start_cell[1]})\n")
        sys.stdout.flush()

        # 3. Drag through intermediate points (slower, more deliberate steps)
        for coord in path[1:]:
            await page.mouse.move(coord[0], coord[1], steps=5)
            await page.wait_for_timeout(self.timing['mouse_move_step_delay'])
            sys.stdout.write(f"       → ({coord[0]},{coord[1]})\n")
            sys.stdout.flush()

        # 4. Release mouse
        await page.mouse.up()
        sys.stdout.write(f"     ✓ Released - word submitted\n")
        sys.stdout.flush()

        # Initial wait after release
        await page.wait_for_timeout(self.timing['after_selection'])
        return True

    async def solve_puzzle(self, page, word_paths):
        sys.stdout.write(f"\n{'='*70}\n")
        sys.stdout.write(f" Starting to solve puzzle with {len(word_paths)} words\n")
        sys.stdout.write(f"{'='*70}\n\n")
        sys.stdout.flush()

        success_count = 0

        # Get initial count
        initial_found = await self.get_current_found_words(page)
        sys.stdout.write(f" Initial theme words found: {initial_found}\n\n")
        sys.stdout.flush()

        for word_num, path in enumerate(word_paths, 1):
            sys.stdout.write(f"\n{'─'*70}\n")
            sys.stdout.write(f"--- Word {word_num}/{len(word_paths)} ---\n")
            sys.stdout.flush()

            # Get current found words before selection
            before_found = await self.get_current_found_words(page)
            sys.stdout.write(f" Theme words before attempt: {before_found}\n")
            sys.stdout.flush()

            drag_success = await self.select_word(page, path)

            if drag_success:
                # Poll to wait until the path turns color (count increases)
                word_accepted = False
                for poll_attempt in range(6):  # Poll up to 6 seconds
                    after_found = await self.get_current_found_words(page)
                    if after_found > before_found:
                        # Determine if blue or yellow based on path length (spangram is longer)
                        if len(path) > 10:
                            sys.stdout.write(f" SUCCESS: Path accepted (turned yellow - spangram).\n")
                        else:
                            sys.stdout.write(f" SUCCESS: Path accepted (turned blue - theme word).\n")
                        sys.stdout.write(f" Theme words after attempt: {after_found} (increased by {after_found - before_found})\n")
                        success_count += 1
                        word_accepted = True
                        break
                    await page.wait_for_timeout(1000)  # Wait 1s before next check

                if not word_accepted:
                    after_found = await self.get_current_found_words(page)
                    sys.stdout.write(f" FAILED: Path not accepted (no color change/no count increase).\n")
                    sys.stdout.write(f" Theme words after attempt: {after_found} (no change)\n")

            # Reset mouse position before next word
            if word_num < len(word_paths):
                await self.reset_mouse_position(page)
                # Apply the increased delay between words
                sys.stdout.write(f"  ⏳ Waiting {self.timing['between_words']/1000}s before next word...\n")
                sys.stdout.flush()
                await page.wait_for_timeout(self.timing['between_words'])

        sys.stdout.write(f"\n{'='*70}\n")
        sys.stdout.write(f"Completed: {success_count}/{len(word_paths)} words successfully selected\n")

        final_found = await self.get_current_found_words(page)
        sys.stdout.write(f" Final theme words found: {final_found}\n")
        sys.stdout.write(f"{'='*70}\n\n")
        sys.stdout.flush()

        try:
            close_button = page.locator('[data-testid="results-modal-close"]')
            if await close_button.is_visible(timeout=5000):
                await close_button.click()
                sys.stdout.write("✓ Closed final results modal.\n")
                await page.wait_for_timeout(self.timing['after_click'])
        except Exception:
            pass

        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        screenshot_path = f"strands_result_{timestamp}.png"

        await page.screenshot(path=screenshot_path)
        sys.stdout.write(f" Final screenshot saved: {screenshot_path}\n")
        sys.stdout.flush()

        return success_count == len(word_paths)

    async def run(self, word_paths, take_screenshot=False, screenshot_path="strands_screenshot.png"):
        if not self.browser:
            await self.start_playwright()

        print("Creating a fresh, isolated browser session...")
        context = await self.browser.new_context(
            viewport={'width': 1920, 'height': 1080},
            user_agent='Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36',
            timezone_id='America/New_York'
        )
        page = await context.new_page()

        try:
            await self.navigate_to_game(page)

            if take_screenshot:
                await self.take_screenshot(page, screenshot_path)

            result = await self.solve_puzzle(page, word_paths)
            return result

        except Exception as e:
            sys.stdout.write(f"\n ERROR: {e}\n")
            sys.stdout.flush()

            traceback.print_exc(file=sys.stdout)

            try:
                await page.screenshot(path="error_screenshot.png")
                sys.stdout.write("📸 Error screenshot saved\n")
                sys.stdout.flush()
            except:
                pass

            return False

        finally:
            await context.close()

    async def stop_playwright(self):
        """Stops Playwright and closes the browser."""
        if self.browser:
            await self.browser.close()
        if self.playwright:
            await self.playwright.stop()
        print("Playwright session stopped.")


async def solve_strands(word_paths, headless=True, board_rows=8, board_cols=6, take_screenshot=False, reset_position=(1059, 161)):
    """Run the Strands automation"""
    automator = StrandsAutomation(headless=headless, board_rows=board_rows, board_cols=board_cols, reset_position=reset_position)
    result = await automator.run(word_paths, take_screenshot=take_screenshot)
    await automator.stop_playwright()
    return result

print("✓ StrandsAutomation class loaded successfully")


✓ StrandsAutomation class loaded successfully


In [ ]:
import sys
from playwright.async_api import async_playwright

# ----------------------------------------------------------------------
# WORD PATHS AND EXECUTION

my_word_paths = [
    [(1227, 484), (1282, 431), (1283, 377), (1339, 377), (1338, 431), (1284, 485), (1283, 539), (1338, 539), (1338, 485)],
    [(1060, 215), (1059, 161), (1116, 161), (1171, 161), (1115, 215), (1171, 215)],
    [(1283, 215), (1228, 215), (1227, 161), (1282, 161), (1338, 161)],
    [(1115, 323), (1061, 323), (1061, 377), (1115, 377), (1116, 431), (1061, 431), (1061, 485), (1115, 485), (1172, 539), (1116, 539), (1059, 539)],
    [(1060, 269), (1115, 269), (1171, 269), (1226, 269), (1283, 269), (1338, 215), (1338, 269), (1338, 322), (1282, 323), (1227, 323), (1172, 323)],
    [(1228, 539), (1171, 485), (1171, 431), (1228, 431), (1171, 377), (1227, 377)],
]

#top-left reset position from your grid coordinates
reset_position = (1059, 161)

sys.stdout.write("="*70 + "\n")
sys.stdout.write("Starting Strands automation...\n")
sys.stdout.write("="*70 + "\n")
sys.stdout.flush()

result = await solve_strands(
    my_word_paths,
    headless=True,
    take_screenshot=True
)

sys.stdout.write("\n" + "="*70 + "\n")
if result:
    sys.stdout.write("Test completed successfully! Puzzle Solved.\n")
else:
    sys.stdout.write("Test completed with issues. Check the console output and 'error_screenshot.png' for details.\n")
sys.stdout.write("="*70 + "\n")
sys.stdout.flush()


Starting Strands automation...
Creating a fresh, isolated browser session...
Navigating to https://www.nytimes.com/games/strands...
Found and clicked the initial 'Play' button.
Waiting for the 'How to Play' tutorial...
Tutorial detected. Navigating...
Clicked 'Next' (1/2).
Clicked 'Next' (2/2).
Clicked final 'Play' button to start the game.
Waiting for the main game area to appear...
✓ Game board is visible and ready.
Taking a focused screenshot of the game area -> strands_screenshot.png
Screenshot successful.

 Starting to solve puzzle with 6 words

 Initial theme words found: 0


──────────────────────────────────────────────────────────────────────
--- Word 1/6 ---
 Theme words before attempt: 0
   Selecting: 9-letter word
     Path: (1227,484) → (1282,431) → (1283,377) → (1339,377) → (1338,431) → (1284,485) → (1283,539) → (1338,539) → (1338,485)
     Start Drag: (1227,484)
       → (1282,431)
       → (1283,377)
       → (1339,377)
       → (1338,431)
       → (1284,485)
       → (